In [3]:
import pandas as pd
import numpy as np

In [4]:
# Carga del dataset

df1: pd.DataFrame = pd.read_csv('data/clean/all_parameters_yearly.csv')
df2: pd.DataFrame = pd.read_csv('data/clean/all_parameters_monthly.csv')
df3: pd.DataFrame = pd.read_csv('data/clean/volumen_util_energia_embalse.csv')

In [6]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 19 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   year                                                     91 non-null     int64  
 1   aportes_hidricos__Value_GWh                              27 non-null     float64
 2   clima_colombia_diario_2000_2026__precipitacion_total_mm  27 non-null     float64
 3   clima_colombia_diario_2000_2026__temp_promedio_c         27 non-null     float64
 4   clima_colombia_diario_2000_2026__temp_maxima_c           27 non-null     float64
 5   cpc_oni_historico__TOTAL                                 77 non-null     float64
 6   cpc_oni_historico__ANOM                                  77 non-null     float64
 7   cpc_roni_historico__ANOM                                 77 non-null     float64
 8   demanda_comercial_reg__Value_GWh           

In [7]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 1092 entries, 0 to 1091
Data columns (total 21 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   year-month                                               1092 non-null   str    
 1   year                                                     1092 non-null   int64  
 2   month                                                    1092 non-null   int64  
 3   aportes_hidricos__Value_GWh                              323 non-null    float64
 4   clima_colombia_diario_2000_2026__precipitacion_total_mm  323 non-null    float64
 5   clima_colombia_diario_2000_2026__temp_promedio_c         323 non-null    float64
 6   clima_colombia_diario_2000_2026__temp_maxima_c           323 non-null    float64
 7   cpc_oni_historico__TOTAL                                 921 non-null    float64
 8   cpc_oni_historico__ANOM                

In [8]:
df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 6870 entries, 0 to 6869
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       6870 non-null   str    
 1   year       6870 non-null   int64  
 2   month      6870 non-null   int64  
 3   Id         6870 non-null   str    
 4   Name       6870 non-null   str    
 5   Value_GWh  6870 non-null   float64
dtypes: float64(1), int64(2), str(3)
memory usage: 322.2 KB


In [9]:
# Código generado por Claude. 
import numpy as np
import pandas as pd

RONI = "cpc_roni_historico__ANOM"
UMBRAL = 0.5
MIN_MESES = 5  # trimestres móviles consecutivos (cada fila mensual = un trimestre)

df2 = df2.sort_values(["year", "month"]).reset_index(drop=True)
df2["fecha"] = pd.to_datetime(df2["year-month"])

# 1. Fase ENSO (NaN donde no hay RONI)
r = df2[RONI]
df2["fase_enso"] = np.select([r >= UMBRAL, r <= -UMBRAL], ["Niño", "Niña"], default="Neutro")
df2.loc[r.isna(), "fase_enso"] = np.nan

# 2. Episodio declarado: rachas >= MIN_MESES en la misma fase
def marcar_episodios(fase: pd.Series, objetivo: str, min_len: int) -> pd.Series:
    mask = fase.eq(objetivo)
    run_id = (mask != mask.shift()).cumsum()
    largo = mask.groupby(run_id).transform("size")
    return mask & (largo >= min_len)

df2["episodio_nino"] = marcar_episodios(df2["fase_enso"], "Niño", MIN_MESES)
df2["episodio_nina"] = marcar_episodios(df2["fase_enso"], "Niña", MIN_MESES)
df2["episodio_enso"] = np.select(
    [df2["episodio_nino"], df2["episodio_nina"]], ["Niño", "Niña"], default="Ninguno"
)

# 3. Intensidad (ordenada), solo sobre |RONI|
etiquetas = ["Neutro", "Débil", "Moderada", "Fuerte", "Muy fuerte"]
df2["intensidad"] = pd.cut(
    r.abs(), bins=[-np.inf, 0.5, 1.0, 1.5, 2.0, np.inf],
    labels=etiquetas, right=False,
)
df2["intensidad"] = pd.Categorical(df2["intensidad"], categories=etiquetas, ordered=True)

# 4. Anomalía de temperatura (z-score por mes calendario)
for col in ["temp_promedio_c", "temp_maxima_c"]:
    c = f"clima_colombia_diario_2000_2026__{col}"
    g = df2.groupby("month")[c]
    df2[f"z_{col}"] = (df2[c] - g.transform("mean")) / g.transform("std")

In [10]:
cols_enso = ["fase_enso", "episodio_enso", "intensidad"]

# ---- df3: volumen útil por embalse (clave: year, month) ----
df3 = df3.drop(columns=cols_enso, errors="ignore")
df3 = df3.merge(df2[["year", "month"] + cols_enso], on=["year", "month"], how="left")
df3["fase_enso"] = pd.Categorical(df3["fase_enso"], categories=["Niña", "Neutro", "Niño"], ordered=True)
df3["intensidad"] = pd.Categorical(df3["intensidad"], categories=etiquetas, ordered=True)

# ---- df1: anual (clave: year) ----
anual = (
    df2.assign(
        es_nino=df2["fase_enso"].eq("Niño"),
        es_nina=df2["fase_enso"].eq("Niña"),
        roni_abs=df2[RONI].abs(),
    )
    .groupby("year")
    .agg(
        meses_nino=("es_nino", "sum"),
        meses_nina=("es_nina", "sum"),
        meses_episodio_nino=("episodio_nino", "sum"),
        meses_episodio_nina=("episodio_nina", "sum"),
        roni_abs_max=("roni_abs", "max"),
        roni_promedio=(RONI, "mean"),
    )
    .reset_index()
)

anual["intensidad_max"] = pd.cut(
    anual["roni_abs_max"], bins=[-np.inf, 0.5, 1.0, 1.5, 2.0, np.inf],
    labels=etiquetas, right=False,
)
anual["intensidad_max"] = pd.Categorical(anual["intensidad_max"], categories=etiquetas, ordered=True)

# Año con al menos un mes de episodio declarado
anual["anio_episodio_nino"] = anual["meses_episodio_nino"] > 0

df1 = df1.drop(columns=[c for c in anual.columns if c != "year"], errors="ignore")
df1 = df1.merge(anual, on="year", how="left")

# Campos calculados para el análisis    
A continuación, código generado por Claude, con campos calculados importantes que normalizan / formatean

In [12]:


# ============ CONFIGURACIÓN ============
RONI = "cpc_roni_historico__ANOM"
P = "clima_colombia_diario_2000_2026__"
UMBRAL, MIN_MESES = 0.5, 5
LAGS_RONI = [1, 3, 6]
ETIQ_INT = ["Neutro", "Débil", "Moderada", "Fuerte", "Muy fuerte"]
BINS_INT = [-np.inf, 0.5, 1.0, 1.5, 2.0, np.inf]
ORDEN_FASE = ["Niña", "Neutro", "Niño"]


def marcar_episodios(fase: pd.Series, objetivo: str, min_len: int) -> pd.Series:
    mask = fase.eq(objetivo)
    run_id = (mask != mask.shift()).cumsum()
    return mask & (mask.groupby(run_id).transform("size") >= min_len)


def intensidad(serie_abs: pd.Series) -> pd.Series:
    out = pd.cut(serie_abs, bins=BINS_INT, labels=ETIQ_INT, right=False)
    return pd.Categorical(out, categories=ETIQ_INT, ordered=True)


# ============ df2: MENSUAL ============
df2 = df2.sort_values(["year", "month"]).reset_index(drop=True)
df2["fecha"] = pd.to_datetime(df2["year-month"])
assert (df2["fecha"].diff().dropna().dt.days.between(28, 31)).all(), "Hay huecos en la serie mensual"

r = df2[RONI]

# 1. Fase, episodio, intensidad
df2["fase_enso"] = np.select([r >= UMBRAL, r <= -UMBRAL], ["Niño", "Niña"], default="Neutro")
df2.loc[r.isna(), "fase_enso"] = np.nan
df2["episodio_nino"] = marcar_episodios(df2["fase_enso"], "Niño", MIN_MESES)
df2["episodio_nina"] = marcar_episodios(df2["fase_enso"], "Niña", MIN_MESES)
df2["episodio_enso"] = np.select([df2["episodio_nino"], df2["episodio_nina"]], ["Niño", "Niña"], default="Ninguno")
df2["intensidad"] = intensidad(r.abs())

# 2. Rezagos del RONI
for k in LAGS_RONI:
    df2[f"roni_lag_{k}"] = r.shift(k)

# 3. Anomalía de temperatura (z-score por mes calendario)
for c in ["temp_promedio_c", "temp_maxima_c"]:
    g = df2.groupby("month")[P + c]
    df2[f"z_{c}"] = (df2[P + c] - g.transform("mean")) / g.transform("std")

# 4. Hidrología
df2["anomalia_aportes"] = df2["aportes_hidricos__Value_GWh"] / df2["media_historica_aportes__Value_GWh"]
df2["anomalia_aportes_pct"] = (df2["anomalia_aportes"] - 1) * 100
vu = df2["volumen_util_energia_sistema__Value_GWh"]
df2["volumen_util_pct_max"] = vu / vu.max() * 100

# 5. Demanda y operación
dem = df2["demanda_real_sin__Value_GWh"]
df2["demanda_real_yoy_pct"] = dem.pct_change(12, fill_method=None) * 100
df2["demanda_vs_tendencia"] = dem / dem.rolling(12, center=True, min_periods=12).mean()
df2["brecha_gen_dem_gwh"] = df2["generacion_real__Value_GWh"] - dem

# 6. Mercado
pb = df2["precio_bolsa_nacional__Value"]
df2["precio_bolsa_log"] = np.log(pb.where(pb > 0))
df2["precio_bolsa_yoy_pct"] = pb.pct_change(12, fill_method=None) * 100

# 7. Proyección UPME vs. real (solo meses con ambos)
up = df2["upme_proyeccion_sin_mensual__demanda_sin_esc_medio_gwh"]
df2["error_upme_pct"] = (dem - up) / up * 100
dentro = (dem >= df2["upme_proyeccion_sin_mensual__ic_inf_95_gwh"]) & \
         (dem <= df2["upme_proyeccion_sin_mensual__ic_sup_95_gwh"])
df2["dentro_ic95"] = dentro.where(up.notna() & dem.notna()).astype("boolean")

df2["fase_enso"] = pd.Categorical(df2["fase_enso"], categories=ORDEN_FASE, ordered=True)

# ============ df3: EMBALSES ============
cols_enso = ["fase_enso", "episodio_enso", "intensidad"]
df3 = df3.drop(columns=cols_enso, errors="ignore")
df3 = df3.merge(df2[["year", "month"] + cols_enso], on=["year", "month"], how="left")
df3["volumen_util_pct_max_embalse"] = df3["Value_GWh"] / df3.groupby("Id")["Value_GWh"].transform("max") * 100

# ============ df1: ANUAL ============
anual = (
    df2.assign(es_nino=df2["fase_enso"].eq("Niño"), es_nina=df2["fase_enso"].eq("Niña"), roni_abs=r.abs())
    .groupby("year")
    .agg(meses_nino=("es_nino", "sum"), meses_nina=("es_nina", "sum"),
         meses_episodio_nino=("episodio_nino", "sum"), meses_episodio_nina=("episodio_nina", "sum"),
         roni_abs_max=("roni_abs", "max"), roni_promedio=(RONI, "mean"))
    .reset_index()
)
anual["intensidad_max"] = intensidad(anual["roni_abs_max"])
anual["anio_episodio_nino"] = anual["meses_episodio_nino"] > 0

df1 = df1.sort_values("year").drop(columns=[c for c in anual.columns if c != "year"], errors="ignore")
df1 = df1.merge(anual, on="year", how="left")
df1["anomalia_aportes"] = df1["aportes_hidricos__Value_GWh"] / df1["media_historica_aportes__Value_GWh"]
df1["demanda_real_yoy_pct"] = df1["demanda_real_sin__Value_GWh"].pct_change(fill_method=None) * 100

# ============ EXPORTACIÓN PARA TABLEAU ============
df2.to_csv("monthly_tableau.csv", index=False, encoding="utf-8-sig")
df3.to_csv("embalses_tableau.csv", index=False, encoding="utf-8-sig")
df1.to_csv("yearly_tableau.csv", index=False, encoding="utf-8-sig")

## Drops hechos para conservar la simpleza
A continuación, una limpieza de campos no necesarios para el reporte


In [15]:
# ============ LIMPIEZA FINAL (ejecutar después del bloque anterior, antes de exportar) ============
P = "clima_colombia_diario_2000_2026__"
UPME = "upme_proyeccion_sin_mensual__"

# Columnas de origen reemplazadas por las derivadas, o redundantes
comunes = [
    "cpc_oni_historico__TOTAL",        # se usa RONI
    "cpc_oni_historico__ANOM",         # se usa RONI
    UPME + "ic_sup_68_gwh",            # se conserva solo el IC 95 %
    UPME + "ic_inf_68_gwh",
    P + "temp_promedio_c",             # se conserva temp_maxima_c y su z-score
    "media_historica_aportes__Value_GWh",  # ya incorporada en anomalia_aportes
]

drop_df2 = comunes + [
    "year-month",                      # reemplazada por fecha
    "episodio_nino", "episodio_nina",  # redundantes con episodio_enso
    "precio_bolsa_log",                # usar escala logarítmica del eje en Tableau
    "brecha_gen_dem_gwh",              # sin interpretación directa (ver advertencia)
    "z_temp_promedio_c",
]
drop_df1 = comunes + ["anio_episodio_nino"] if False else comunes  # df1 conserva todo lo demás

df2_rep = df2.drop(columns=drop_df2, errors="ignore")
df1_rep = df1.drop(columns=drop_df1, errors="ignore")
df3_rep = df3  # sin cambios

# Reordenar: tiempo y clasificación primero
def ordenar(df, primeras):
    primeras = [c for c in primeras if c in df.columns]
    return df[primeras + [c for c in df.columns if c not in primeras]]

df2_rep = ordenar(df2_rep, ["fecha", "year", "month", "fase_enso", "episodio_enso", "intensidad", RONI])
df1_rep = ordenar(df1_rep, ["year", "meses_nino", "meses_nina", "intensidad_max", "roni_promedio"])

# ============ EXPORTACIÓN ============
df2_rep.to_csv("monthly_tableau.csv", index=False, encoding="utf-8-sig")
df3_rep.to_csv("embalses_tableau.csv", index=False, encoding="utf-8-sig")
df1_rep.to_csv("yearly_tableau.csv", index=False, encoding="utf-8-sig")

In [16]:

UMBRAL_SEQUIA = 1.0   # < 1 = aportes bajo la media histórica; endurécelo (p. ej. 0.8) si quieres sequía más marcada

# df2_rep (mensual)
a = df2_rep["anomalia_aportes"]
df2_rep["sequia"] = (a < UMBRAL_SEQUIA).where(a.notna()).astype("boolean")

# df3_rep (embalses): hereda por mes
df3_rep = df3_rep.drop(columns="sequia", errors="ignore").merge(
    df2_rep[["year", "month", "sequia"]], on=["year", "month"], how="left"
)

# df1_rep (anual): meses en sequía por año
meses_sequia = df2_rep.groupby("year")["sequia"].sum().rename("meses_sequia").reset_index()
df1_rep = df1_rep.drop(columns="meses_sequia", errors="ignore").merge(meses_sequia, on="year", how="left")

# Re-exportar
df2_rep.to_csv("monthly_tableau.csv", index=False, encoding="utf-8-sig")
df3_rep.to_csv("embalses_tableau.csv", index=False, encoding="utf-8-sig")
df1_rep.to_csv("yearly_tableau.csv", index=False, encoding="utf-8-sig")

In [ ]:
df1.to_csv("data/clean/all_parameters_yearly.csv", index=False)
df2.to_csv("data/clean/all_parameters_monthly.csv", index=False)
df3.to_csv("data/clean/volumen_util_energia_embalse.csv", index=False)